# Direct deployment with vLLM (no AirLLM)

Gemma 4 (E2B-qat, E4B, 12B, 26B-A4B) and Qwen 3.5-9B / 3.6-27B / 3.8-27B on one big GPU. bf16 when the checkpoint fits in 75 % of VRAM, else vLLM fp8. Generated by `scripts/make_direct_notebook.py` in the airllm-benchmarks repo.

In [ ]:
!nvidia-smi
!df -h / /tmp /teamspace/studios/this_studio 2>/dev/null
!free -g
!nproc

In [ ]:
%%capture pip_log
!pip install -q 'vllm==0.34.0' nvidia-ml-py psutil 'huggingface_hub[hf_xet]'

In [ ]:
import torch, transformers, vllm
print('torch', torch.__version__, 'cuda', torch.version.cuda, torch.cuda.get_device_name(0))
print('transformers', transformers.__version__, 'vllm', vllm.__version__)

In [ ]:
import os
os.makedirs('airllm-benchmarks/bench', exist_ok=True)

In [ ]:
%%writefile airllm-benchmarks/bench/measure.py
"""Measurement helpers shared by the AirLLM runner (run_bench.py) and the direct-deployment
runner (run_direct.py): memory sampler, per-token timer, chat-template prompt builder."""

from __future__ import annotations

import os
import threading
import time

import psutil
import torch

DEFAULT_PROMPT = "Explain how a CPU cache works in one short paragraph."


class Sampler(threading.Thread):
    """Samples process RSS and GPU memory (NVML, whole device and this process) every 100 ms."""

    def __init__(self, interval=0.1):
        super().__init__(daemon=True)
        self.interval = interval
        self.proc = psutil.Process()
        self._stop = threading.Event()
        self.reset()
        try:
            import pynvml

            pynvml.nvmlInit()
            self.nvml = pynvml
            self.handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        except Exception:  # noqa: BLE001
            self.nvml = None

    def reset(self):
        self.peak_rss = 0
        self.peak_gpu_device = 0
        self.peak_gpu_process = 0
        self.other_gpu_pids = set()

    def gpu(self):
        if self.nvml is None:
            return 0, 0
        dev = self.nvml.nvmlDeviceGetMemoryInfo(self.handle).used
        mine = 0
        try:
            for p in self.nvml.nvmlDeviceGetComputeRunningProcesses(self.handle):
                if p.pid == os.getpid():
                    mine = p.usedGpuMemory or 0
                elif (p.usedGpuMemory or 0) > 256 * 2**20:
                    self.other_gpu_pids.add(p.pid)  # another workload shared the GPU
        except Exception:  # noqa: BLE001
            pass
        return dev, mine

    def run(self):
        while not self._stop.is_set():
            try:
                self.peak_rss = max(self.peak_rss, self.proc.memory_info().rss)
                dev, mine = self.gpu()
                self.peak_gpu_device = max(self.peak_gpu_device, dev)
                self.peak_gpu_process = max(self.peak_gpu_process, mine)
            except Exception:  # noqa: BLE001
                pass
            time.sleep(self.interval)

    def stop(self):
        self._stop.set()

    def snapshot(self):
        return {
            "peak_rss_gb": round(self.peak_rss / 1e9, 3),
            "peak_gpu_device_used_gb": round(self.peak_gpu_device / 1e9, 3),
            "peak_gpu_process_gb": round(self.peak_gpu_process / 1e9, 3),
            "other_gpu_pids_seen": sorted(self.other_gpu_pids),
        }


class TokenTimer:
    """LogitsProcessor that timestamps each decoding step (called once per generated token)."""

    def __init__(self):
        self.times = []

    def __call__(self, input_ids, scores):
        torch.cuda.synchronize()
        self.times.append(time.perf_counter())
        return scores


def build_inputs(tokenizer, prompt):
    messages = [{"role": "user", "content": prompt}]
    text = None
    if getattr(tokenizer, "chat_template", None):
        for kw in ({"enable_thinking": False}, {}):
            try:
                text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **kw)
                break
            except Exception:  # noqa: BLE001
                continue
    used_template = text is not None
    if text is None:
        text = prompt
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=not used_template)
    return enc, used_template, text


In [ ]:
%%writefile airllm-benchmarks/bench/model_list.py
"""Benchmarked models: short name -> Hugging Face repo id (no AirLLM import, so the vLLM and
ZeroGPU runners can use it without installing AirLLM)."""

MODELS = {
    "gemma-4-E2B-it-qat-mobile-ct": "google/gemma-4-E2B-it-qat-mobile-ct",
    "gemma-4-E4B-it": "google/gemma-4-E4B-it",
    "gemma-4-12B-it": "google/gemma-4-12B-it",
    "gemma-4-26B-A4B-it": "google/gemma-4-26B-A4B-it",
    "Qwen3.5-9B": "Qwen/Qwen3.5-9B",
    "Qwen3.6-27B": "Qwen/Qwen3.6-27B",
    "Qwen3.8-27B": "Qwen/Qwen3.8-27B",
}


In [ ]:
%%writefile airllm-benchmarks/bench/run_direct.py
#!/usr/bin/env python
"""Benchmark one model deployed directly on a GPU that holds all of it (no AirLLM); write JSON.

Engines
  vllm  vLLM's offline engine: bf16 weights (or ``--quantization fp8`` when bf16 does not fit),
        torch.compile + CUDA graphs, prefix caching off, multimodal encoders disabled.
  hf    plain transformers ``generate()`` with the whole model on one GPU (eager, dynamic cache).
        Used on Hugging Face ZeroGPU, which runs PyTorch inside a Gradio Space but not vLLM.

Same prompt, chat template (thinking off) and greedy decoding as the AirLLM runs, measured as
  single  one request: time to first token (prefill) and decode tok/s (median of --repeats)
  batch   --batch identical requests x --batch-tokens tokens each: aggregate output tok/s (vllm)
Download and engine start-up (weights load, compile, CUDA-graph capture) are timed separately.
"""

from __future__ import annotations

import argparse
import json
import logging
import os
import platform
import re
import socket
import statistics
import sys
import time
import traceback
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))

from measure import DEFAULT_PROMPT, Sampler, TokenTimer, build_inputs  # noqa: E402
from model_list import MODELS  # noqa: E402

ALLOW = ["*.json", "*.safetensors", "*.model", "*.jinja", "*.txt"]


# ---------------------------------------------------------------------------------------------
# environment


def host_info():
    import psutil

    info = {
        "hostname_tag": os.environ.get("BENCH_HOST_TAG", socket.gethostname()),
        "platform": platform.platform(),
        "python": platform.python_version(),
        "cpu": platform.processor() or "",
        "cpu_count": psutil.cpu_count(),
        "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 1),
    }
    try:
        with open("/proc/cpuinfo") as fh:
            for line in fh:
                if line.startswith("model name"):
                    info["cpu"] = line.split(":", 1)[1].strip()
                    break
    except OSError:
        pass
    from importlib.metadata import version

    for pkg in ("torch", "transformers", "vllm", "compressed-tensors", "flashinfer-python", "spaces", "gradio"):
        try:
            info[pkg.replace("-", "_")] = version(pkg)
        except Exception:  # noqa: BLE001
            pass
    return info


def gpu_info():
    """Needs a real CUDA context (on ZeroGPU: call inside the @spaces.GPU function)."""
    import torch

    if not torch.cuda.is_available():
        return {}
    p = torch.cuda.get_device_properties(0)
    return {
        "gpu": p.name,
        "gpu_vram_gb": round(p.total_memory / 1e9, 2),
        "gpu_cc": f"{p.major}.{p.minor}",
        "gpu_count": torch.cuda.device_count(),
        "cuda_runtime": torch.version.cuda,
        "bf16_native": torch.cuda.is_bf16_supported(including_emulation=False),
    }


def download(repo_id):
    from huggingface_hub import snapshot_download

    t = time.perf_counter()
    path = Path(snapshot_download(repo_id, allow_patterns=ALLOW))
    size = sum(f.stat().st_size for f in path.glob("*.safetensors"))
    return path, round(time.perf_counter() - t, 2), round(size / 1e9, 3)


def modalities(path):
    """Multimodal inputs the checkpoint declares; vLLM gets a limit of 0 for each (text only)."""
    cfg = json.loads((Path(path) / "config.json").read_text())
    mods = []
    if cfg.get("vision_config"):
        mods.append("image")
        if cfg.get("model_type", "").startswith("qwen"):
            mods.append("video")
    if cfg.get("audio_config"):
        mods.append("audio")
    return mods


# ---------------------------------------------------------------------------------------------
# vLLM


class _LogTap(logging.Handler):
    def __init__(self):
        super().__init__()
        self.lines = []

    def emit(self, record):
        try:
            self.lines.append(record.getMessage())
        except Exception:  # noqa: BLE001
            pass


_VLLM_PATTERNS = {
    "weights_gib": (r"Model loading took ([\d.]+) ?GiB", float),
    "weights_load_s": (r"Model loading took [\d.]+ ?GiB(?: memory)? and ([\d.]+) ?s", float),
    "kv_cache_gib": (r"Available KV cache memory: ([\d.]+) ?GiB", float),
    "kv_cache_tokens": (r"GPU KV cache size: ([\d,]+) tokens", lambda s: int(s.replace(",", ""))),
    "max_concurrency": (r"Maximum concurrency for [\d,]+ tokens per request: ([\d.]+)x", float),
    "compile_s": (r"torch\.compile takes ([\d.]+) ?s in total", float),
    "graph_capture_s": (r"Graph capturing finished in ([\d.]+) ?sec", float),
    "attention_backend": (r"Using ([A-Z_]+[A-Z0-9_]*) (?:attention )?backend", str),
}


def parse_vllm_log(lines):
    out = {}
    for key, (pat, conv) in _VLLM_PATTERNS.items():
        for line in lines:
            m = re.search(pat, line)
            if m:
                try:
                    out[key] = conv(m.group(1))
                except ValueError:
                    pass
                break
    return out


def vllm_bench(path, args, result, sampler):
    # engine core in this process: its logs reach our handler and NVML per-process memory is ours
    os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    tap = _LogTap()
    logging.getLogger("vllm").addHandler(tap)

    tok = AutoTokenizer.from_pretrained(path)
    enc, used_template, _ = build_inputs(tok, args.prompt)
    prompt = {"prompt_token_ids": enc["input_ids"][0].tolist()}
    result["chat_template"] = used_template

    kw = dict(model=str(path), dtype=args.dtype, max_model_len=args.max_model_len,
              gpu_memory_utilization=args.gpu_mem, enable_prefix_caching=False, seed=0)
    if args.quantization:
        kw["quantization"] = args.quantization
    mods = modalities(path)
    if mods:
        kw["limit_mm_per_prompt"] = {m: 0 for m in mods}
    result["engine_args"] = {k: v for k, v in kw.items() if k != "model"}

    sampler.reset()
    t = time.perf_counter()
    llm = LLM(**kw)
    result["phases"]["load_s"] = round(time.perf_counter() - t, 2)
    result["phases"]["load_mem"] = sampler.snapshot()

    def run(n, batch=1, ignore_eos=True):
        sp = SamplingParams(temperature=0.0, max_tokens=n, ignore_eos=ignore_eos)
        t0 = time.perf_counter()
        outs = llm.generate([prompt] * batch, sp, use_tqdm=False)
        return time.perf_counter() - t0, outs

    run(8)  # warm-up
    sampler.reset()
    ttfts = [run(1)[0] for _ in range(args.repeats)]
    totals = [run(args.tokens)[0] for _ in range(args.repeats)]
    ttft, total = statistics.median(ttfts), statistics.median(totals)
    _, outs = run(32, ignore_eos=False)
    single = {
        "prompt_tokens": len(prompt["prompt_token_ids"]),
        "new_tokens": args.tokens,
        "ttft_s": round(ttft, 4),
        "ttft_samples_s": [round(x, 4) for x in ttfts],
        "total_s": round(total, 4),
        "total_samples_s": [round(x, 4) for x in totals],
        "decode_tok_per_s": round((args.tokens - 1) / (total - ttft), 2),
        "end_to_end_tok_per_s": round(args.tokens / total, 2),
        "text": outs[0].outputs[0].text,
    }
    single.update(sampler.snapshot())
    result["runs"]["single"] = single
    print(json.dumps(single, indent=1), flush=True)

    if args.batch > 1:
        sampler.reset()
        dt, outs = run(args.batch_tokens, batch=args.batch)
        n_out = sum(len(o.outputs[0].token_ids) for o in outs)
        batch = {"batch": args.batch, "tokens_each": args.batch_tokens, "total_s": round(dt, 3),
                 "output_tokens": n_out, "output_tok_per_s": round(n_out / dt, 1)}
        batch.update(sampler.snapshot())
        result["runs"]["batch"] = batch
        print(json.dumps(batch, indent=1), flush=True)

    result["engine_info"] = parse_vllm_log(tap.lines)
    result["engine_log_tail"] = tap.lines[-60:]


# ---------------------------------------------------------------------------------------------
# plain transformers (also imported by the ZeroGPU Space)


def hf_load(path, dtype="bfloat16", device_map="cuda"):
    """Whole model on one device. Multimodal checkpoints load with their image-text class so that
    every language-model weight lands where the checkpoint has it (no silently missing keys)."""
    import torch
    from transformers import AutoModelForCausalLM, AutoModelForImageTextToText, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(path)
    cls = AutoModelForImageTextToText if modalities(path) else AutoModelForCausalLM
    model, info = cls.from_pretrained(path, dtype=getattr(torch, dtype), device_map=device_map,
                                      output_loading_info=True)
    missing = [k for k in info.get("missing_keys", []) if "lm_head" not in k]
    if missing:
        raise RuntimeError(f"{len(missing)} weights missing after load, e.g. {missing[:3]}")
    model.eval()
    return model, tok, cls.__name__


def hf_measure(model, tok, prompt=DEFAULT_PROMPT, n_tokens=64, repeats=2, text_tokens=32):
    import torch
    from transformers import LogitsProcessorList

    enc, used_template, _ = build_inputs(tok, prompt)
    dev = next(model.parameters()).device
    ids, attn = enc["input_ids"].to(dev), enc["attention_mask"].to(dev)
    gen = dict(do_sample=False, temperature=None, top_p=None, top_k=None, use_cache=True,
               pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
    with torch.inference_mode():
        model.generate(ids, attention_mask=attn, max_new_tokens=4, **gen)  # warm-up
        torch.cuda.reset_peak_memory_stats()
        runs = []
        for _ in range(repeats):
            timer = TokenTimer()
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            out = model.generate(ids, attention_mask=attn, max_new_tokens=n_tokens, min_new_tokens=n_tokens,
                                 logits_processor=LogitsProcessorList([timer]), **gen)
            torch.cuda.synchronize()
            t1 = time.perf_counter()
            ts = timer.times
            runs.append({"ttft_s": ts[0] - t0, "decode_tok_per_s": (len(ts) - 1) / (ts[-1] - ts[0]),
                         "total_s": t1 - t0, "n": len(ts)})
    new = out[0, ids.shape[1]:]
    return {
        "chat_template": used_template,
        "prompt_tokens": int(ids.shape[1]),
        "new_tokens": runs[-1]["n"],
        "ttft_s": round(statistics.median(r["ttft_s"] for r in runs), 4),
        "ttft_samples_s": [round(r["ttft_s"], 4) for r in runs],
        "decode_tok_per_s": round(statistics.median(r["decode_tok_per_s"] for r in runs), 2),
        "decode_samples_tok_per_s": [round(r["decode_tok_per_s"], 2) for r in runs],
        "total_s": round(statistics.median(r["total_s"] for r in runs), 4),
        "torch_peak_allocated_gb": round(torch.cuda.max_memory_allocated() / 1e9, 3),
        "text": tok.decode(new[:text_tokens], skip_special_tokens=True),
    }


def hf_bench(path, args, result, sampler):
    sampler.reset()
    t = time.perf_counter()
    model, tok, cls = hf_load(path, dtype=args.dtype if args.dtype != "auto" else "bfloat16")
    result["phases"]["load_s"] = round(time.perf_counter() - t, 2)
    result["phases"]["load_mem"] = sampler.snapshot()
    result["hf_class"] = cls
    sampler.reset()
    single = hf_measure(model, tok, args.prompt, n_tokens=args.tokens, repeats=args.repeats)
    single.update(sampler.snapshot())
    result["chat_template"] = single.pop("chat_template")
    result["runs"]["single"] = single
    print(json.dumps(single, indent=1), flush=True)


# ---------------------------------------------------------------------------------------------


def result_name(short, engine, quantization):
    return f"{short}__{engine}-{quantization or 'bf16'}"


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model", required=True, help="short name from bench/model_list.py or a HF repo id")
    ap.add_argument("--engine", default="vllm", choices=["vllm", "hf"])
    ap.add_argument("--quantization", default=None, help="vllm only, e.g. fp8 (on-the-fly, W8A16 on Ampere)")
    ap.add_argument("--dtype", default="auto")
    ap.add_argument("--tokens", type=int, default=256, help="decode length for the single-stream run")
    ap.add_argument("--repeats", type=int, default=3)
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--batch-tokens", type=int, default=256)
    ap.add_argument("--max-model-len", type=int, default=4096)
    ap.add_argument("--gpu-mem", type=float, default=0.90)
    ap.add_argument("--prompt", default=DEFAULT_PROMPT)
    ap.add_argument("--out-dir", default=str(HERE.parent / "results" / "direct"))
    args = ap.parse_args()

    repo_id = MODELS.get(args.model, args.model)
    short = args.model if args.model in MODELS else repo_id.split("/")[-1]
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / f"{result_name(short, args.engine, args.quantization)}.json"

    result = {
        "model": short, "repo_id": repo_id, "engine": args.engine,
        "weights": args.quantization or "bf16", "status": "running",
        "started": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "prompt": args.prompt, "phases": {}, "runs": {}, "env": host_info(),
    }
    sampler = Sampler()
    sampler.start()
    phase = "download"
    try:
        path, result["phases"]["download_s"], result["phases"]["checkpoint_gb"] = download(repo_id)
        phase = "load+generate"
        (vllm_bench if args.engine == "vllm" else hf_bench)(path, args, result, sampler)
        result["env"].update(gpu_info())
        result["status"] = "ok"
    except BaseException as e:  # noqa: BLE001 - record OOMs, KeyboardInterrupt too
        result["status"] = "error"
        result["error_phase"] = phase
        result["error"] = f"{type(e).__name__}: {e}"[:2000]
        result["traceback"] = traceback.format_exc()[-6000:]
        print(result["traceback"], file=sys.stderr, flush=True)
        try:
            result["env"].update(gpu_info())
        except Exception:  # noqa: BLE001
            pass
    finally:
        sampler.stop()
        result["finished"] = time.strftime("%Y-%m-%dT%H:%M:%S")
        out_file.write_text(json.dumps(result, indent=2))
        print(f"wrote {out_file} status={result['status']}", flush=True)
    # vLLM leaves non-daemon threads behind in-process; exit hard so the matrix moves on
    os._exit(0 if result["status"] == "ok" else 1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile airllm-benchmarks/bench/run_direct_matrix.py
#!/usr/bin/env python
"""Run bench/run_direct.py for every model on one big-GPU host, one subprocess per model.

  --fit auto     bf16 when the checkpoint uses at most --fit-frac of VRAM, otherwise vLLM's
                 on-the-fly fp8 (W8A16 Marlin on Ampere, W8A8 on Hopper); hf engine: skip
  --cleanup      delete each model's HF cache after its run (scratch disks are small)
  --budget-min   start no new model after this many minutes (paid GPU credits)

Example (an H100 box):
  BENCH_HOST_TAG=my-h100 python bench/run_direct_matrix.py --out-dir results/my-h100 --cleanup
"""

from __future__ import annotations

import argparse
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))

from model_list import MODELS  # noqa: E402
from run_direct import result_name  # noqa: E402


def vram_gb():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True, check=True).stdout
    return int(out.splitlines()[0]) * 2**20 / 1e9


def checkpoint_gb(repo_id):
    from huggingface_hub import HfApi

    info = HfApi().model_info(repo_id, files_metadata=True)
    return sum(s.size or 0 for s in info.siblings if s.rfilename.endswith(".safetensors")) / 1e9


def cache_dir(repo_id):
    from huggingface_hub import constants

    return Path(constants.HF_HUB_CACHE) / ("models--" + repo_id.replace("/", "--"))


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--engine", default="vllm", choices=["vllm", "hf"])
    ap.add_argument("--models", default="all", help="comma-separated short names, or 'all'")
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--fit", default="auto", choices=["auto", "bf16", "fp8"])
    ap.add_argument("--fit-frac", type=float, default=0.75)
    ap.add_argument("--cleanup", action="store_true")
    ap.add_argument("--budget-min", type=float, default=0, help="0 = no limit")
    ap.add_argument("--timeout-min", type=float, default=40)
    ap.add_argument("extra", nargs="*", help="passed through to run_direct.py (after --)")
    args = ap.parse_args()

    models = list(MODELS) if args.models == "all" else args.models.split(",")
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    log_dir = HERE.parent / "logs"
    log_dir.mkdir(exist_ok=True)
    vram = vram_gb()
    t_start = time.time()
    print(f"[matrix] {len(models)} models, engine={args.engine}, VRAM {vram:.1f} GB", flush=True)

    for short in models:
        repo_id = MODELS[short]
        elapsed = (time.time() - t_start) / 60
        if args.budget_min and elapsed > args.budget_min:
            print(f"[matrix] budget of {args.budget_min} min used ({elapsed:.0f} min): stopping before {short}")
            break
        size = checkpoint_gb(repo_id)
        quant = None
        if args.fit == "fp8" or (args.fit == "auto" and size > args.fit_frac * vram):
            if args.engine != "vllm":
                print(f"[matrix] skip {short}: {size:.1f} GB checkpoint does not fit {vram:.0f} GB in bf16")
                continue
            quant = "fp8"
        res_file = out_dir / f"{result_name(short, args.engine, quant)}.json"
        if res_file.exists() and json.loads(res_file.read_text()).get("status") == "ok":
            print(f"[matrix] done already: {res_file.name}")
            continue
        cmd = [sys.executable, str(HERE / "run_direct.py"), "--model", short, "--engine", args.engine,
               "--out-dir", str(out_dir), *args.extra]
        if quant:
            cmd += ["--quantization", quant]
        log = log_dir / f"direct_{res_file.stem}.log"
        print(f"[matrix] {time.strftime('%H:%M:%S')} {short}: {size:.1f} GB checkpoint -> "
              f"{args.engine} {quant or 'bf16'} (log {log.name})", flush=True)
        t = time.time()
        with open(log, "w") as fh:
            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            try:
                for line in proc.stdout:
                    fh.write(line)
                    if any(k in line for k in ("decode_tok_per_s", "output_tok_per_s", "ttft_s\"", "wrote ",
                                               "Error", "error", "Traceback")):
                        print("   ", line.rstrip()[:300], flush=True)
                    if time.time() - t > args.timeout_min * 60:
                        proc.kill()
                        print(f"[matrix] timeout after {args.timeout_min} min", flush=True)
                        break
            finally:
                proc.wait()
        print(f"[matrix] {short} finished in {(time.time() - t) / 60:.1f} min (exit {proc.returncode})", flush=True)
        if args.cleanup:
            shutil.rmtree(cache_dir(repo_id), ignore_errors=True)
            print(f"[matrix] {shutil.disk_usage(cache_dir(repo_id).parent).free / 1e9:.0f} GB free after cleanup")
    print(f"[matrix] all done in {(time.time() - t_start) / 60:.1f} min", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
import os, re, shutil, torch
%cd airllm-benchmarks
gpu = torch.cuda.get_device_name(0)
m = re.search(r'(H200|H100|A100|L40S|L4|A10G|T4|B200|RTX PRO 6000)', gpu, re.I)
TAG = 'jupyter-' + (m.group(1).lower().replace(' ', '') if m else 'gpu')
cands = [c for c in ['/tmp', '/cache', '/root', os.path.expanduser('~')] if os.path.isdir(c)]
for c in cands: print(c, round(shutil.disk_usage(c).free / 1e9), 'GB free')
SCRATCH = max(cands, key=lambda c: shutil.disk_usage(c).free)
os.environ['HF_HOME'] = f'{SCRATCH}/hf'
os.environ['BENCH_HOST_TAG'] = TAG
print('GPU', gpu, '-> tag', TAG, '| HF_HOME', os.environ['HF_HOME'])

In [ ]:
!python bench/run_direct_matrix.py --models all --out-dir results/{TAG} --cleanup --budget-min 40

In [ ]:
import json, glob
for f in sorted(glob.glob(f'results/{TAG}/*.json')):
    r = json.load(open(f)); s = r['runs'].get('single', {}); b = r['runs'].get('batch', {})
    print(f"{r['model']:30s} {r['engine']}-{r['weights']:5s} {r['status']:5s} load {r['phases'].get('load_s')}s "
          f"ttft {s.get('ttft_s')}s decode {s.get('decode_tok_per_s')} tok/s batch {b.get('output_tok_per_s')} tok/s "
          f"| {r.get('error', '')[:150]}")

In [ ]:
import json, glob
for f in sorted(glob.glob(f'results/{TAG}/*.json')):
    r = json.load(open(f)); r.pop('engine_log_tail', None)
    print('@@@RESULT', f.split('/')[-1]); print(json.dumps(r)); print('@@@END')